# tkh-phase-2-project - Financial Fraud Detection - Explore notebook

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

### Load data

In [ ]:
df_orig = pd.read_csv('../data/PS_20174392719_1491204439457_log.csv')

### Summary stats

In [ ]:
df_orig.info()

Above, we see that we have 11 columns, consisting of 10 features/predictors and one target (`isFraud`). We also see that we have 6.36 million rows and no null/missing values.

In [ ]:
df_orig.describe()

### Rename columns for ease of readability

In [ ]:
df = df_orig.copy()
df.columns.to_list()

In [ ]:
df.columns = [
    'step',
    'type',
    'amount',
    'name_orig',
    'old_balance_orig',
    'new_balance_orig',
    'name_dest',
    'old_balance_dest',
    'new_balance_dest',
    'is_fraud',
    'is_flagged_fraud'
]

### Numeric features

In [ ]:
df_num = df.select_dtypes(include=np.number)
df_num.head()

In [ ]:
df_num.describe()

Above, we see that minimum of the `amount` feature is zero. Transactions of zero money don't seem like valid transactions. Keeping them in the dataset can make it hard to apply log transforms (because log(0) equals neg infinity) so let's inspect how many rows have a zero `amount`.

In [ ]:
df_amount_zero = df_num[df_num['amount'] == 0]
df_amount_zero.info()

In [ ]:
df_amount_zero

Above, we see that there are 16 rows where `amount` equals zero. Since all 16 rows are labeled as fraud, let's keep the rows for now.

In [ ]:
df_num['is_fraud'].value_counts()

Above, we see that just how small our minority class is. There are 8,213 fraud transactions and 6.35 million non-fraud transactions.

In [ ]:
df_num['step'].value_counts()

It's not clear how useful the `step` feature will be. It's an integer between (1, 743) and it represents when the point in time the transaction occurred, with larger values signifying that a transaction occurred later. By looking at the `value_counts()`, we see that some steps had 40-50K transactions, while others had under 5 transactions.

### Non-numeric features

In [ ]:
df_non_num = df.select_dtypes(exclude=np.number)
df_non_num

In [ ]:
df_non_num['type'].value_counts()

The `type` feature contains a small number of values that could have predictive power, so let's encode this column with `get_dummies`.

In [ ]:
df_clean = pd.get_dummies(df_non_num['type'], drop_first=True)
df_clean.columns = [c.lower() for c in df_clean.columns.to_list()]
df_clean

In [ ]:
df_non_num['name_orig'].value_counts()

In [ ]:
df_non_num['name_dest'].value_counts()

Above, we see that `values_counts()` of the `nameOrig` and `nameDest` features. It's not clear how helpful they will be for our predictions. It is concievable that some accounts are more prone to fraud, so perhaps it is helpful for the model to know which accounts the transactions affected.

## Univariate analysis

In [ ]:
columns = df_num.columns.to_list()
columns

In [ ]:
columns = [
    'step',
    'amount',
    'old_balance_orig',
    'new_balance_orig',
    'old_balance_dest',
    'new_balance_dest',
]

In [ ]:
nrows = 2
ncols = 3
title = 'Non-fraud transactions'

figure, axes = plt.subplots(nrows, ncols, figsize=(16, 8))
figure.suptitle(title)
for col, ax in zip(columns, axes.flatten()):
    df_num[df_num['is_fraud'] == 0][col].plot.hist(ax=ax, bins=20, title=col)

In [ ]:
nrows = 2
ncols = 3
title = 'Fraud transactions'

figure, axes = plt.subplots(nrows, ncols, figsize=(16, 8))
figure.suptitle(title)
for col, ax in zip(columns, axes.flatten()):
    df_num[df_num['is_fraud'] == 1][col].plot.hist(ax=ax, bins=20, title=col, color='firebrick')

The histograms above show that our account balance features for all transactions (fraud and non-fraud) are very skewed, with peaks in values between 0 and 2,500,000 in currency units. For the step feature, our histograms are mostly uniformly distributed, more so with the fraud transactions.

## Bivariate analysis

In [ ]:
corr = df_num.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap')

Our correlation heatmap above show extreme collinearity between the old and new balances for both the origin and destination accounts. This makes sense because each transaction affects the two accounts by the same amount in opposite directions. 

We also see some but not as much collinearity between amount and the old/new balances for the destination account. This makes sense because amount equals how much the destination account changes in the transaction. It's unclear why there isn't collinearity between amount and the old/new balancaes for the origin account.

Another noteworthy observation is that there is very little correlation between our features and our target (is_fraud). The column with most correlation with our target is amount, with a value of 0.08 

Before training our models, we'll want to drop half of the feature pairs that showed collinearity. In order words, we'll drop:
* old_balance_orig
* old_balance_dest